In [ ]:
# Biogeme
from bio_estimation_fcns import estimate_mnl, simulate_mnl, estimate_mxl, simulate_mxl, print_results, plot_distributions
import biogeme.biogeme as bio
import biogeme.database as db
from biogeme import models
import biogeme.biogeme_logging as blog
from biogeme.expressions import Beta, Variable, exp, bioDraws

# General python packages
import importlib   
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from pathlib import Path
import os

# Pandas setting to show all columns when displaying a pandas dataframe
pd.set_option('display.max_columns', None)

In [ ]:
# Initialize the logger, if it has not been initialized yet
try:
    logger
except NameError:    
    logger = blog.get_screen_logger(level=blog.INFO)
    print('Logger has been initialised')

#### Load the data

In [ ]:
data_path =  Path(os.getcwd()) / 'data'
print(data_path)

In [ ]:
# Load the data as a pandas dataframe
df = pd.read_csv(data_path / 'df_choice_with_Vimg_hp.csv')
df.drop(columns=['prob','Pchosen','LLn'], inplace=True)
df.dropna(inplace=True)
df.shape

#### Ensure fixed number of observations per individual

In [ ]:
## Fixing RID 63, who made the survey twice (??)

# Boolean mask for the rows you are interested in
mask = df['RID'] == 63

# Index labels of the *last* 15 such rows (ordered as they currently appear)
last_15_idx = df[mask].tail(15).index

# Over-write their RID
df.loc[last_15_idx, 'RID'] = 63999

In [ ]:
print(df['RID'].value_counts().value_counts())

In [ ]:
# Drop RIDs with less than 15 observations
min_obs = 15
print(f'Dropping {df[df["RID"].map(df["RID"].value_counts()) < min_obs]["RID"].nunique()} RIDs with less than {min_obs} observations')
df = df[df["RID"].map(df["RID"].value_counts()) >= min_obs]

#### Check for nontraders

In [ ]:
df_nontr_TT = df[['RID','TL1','TT1','TL2','TT2','CHOICE']].where(((df['TT1'] <= df['TT2']) & (df['CHOICE'] == 1)) | ((df['TT2'] <= df['TT1']) & (df['CHOICE'] == 2))).dropna()
nontr_TT = (df_nontr_TT.groupby('RID').size() == 15)
print(f'{nontr_TT.sum()} respondents are nontraders as they always have chosen the fastest route')

df_nontr_TL = df[['RID','TL1','TT1','TL2','TT2','CHOICE']].where(((df['TL1'] <= df['TL2']) & (df['CHOICE'] == 1)) | ((df['TL2'] <= df['TL1']) & (df['CHOICE'] == 2))).dropna()
nontr_TL = (df_nontr_TL.groupby('RID').size() == 15)
print(f'{nontr_TL.sum()} respondents are nontraders as they always have chosen the route with the least traffic lights')

# Create a list of respondent IDs that are nontraders for TT and TL
rid_nontr = nontr_TT[nontr_TT].index.to_list() + nontr_TL[nontr_TL].index.to_list()

In [ ]:
drop_nontraders = False
if drop_nontraders:
    print(f'Removing {len(rid_nontr)} nontraders from the dataset')

    # Remove nontraders from the dataset
    df = df[~df['RID'].isin(rid_nontr)]
    print(f'Remaining number of respondents after removing nontraders: {df["RID"].nunique()}')
    print(f'Remaining number of observations after removing nontraders: {df.shape[0]}')

else:
    print(f'Not removing nontraders. Keeping {len(rid_nontr)} nontraders in the dataset')

#### Split the data into train and test sets

In [ ]:
# Split
df_train = df[df['train'] == 1].copy()
df_test = df[df['test'] == 1].copy()
df.head()

In [ ]:
print(f'Shape of train data {df_train.shape}\nShape of test data {df_test.shape}\nShare of test data {100*(df_test.shape[0] / (df_train.shape[0]+df_test.shape[0])):0.0f}%')
print(f'There are {df_train["RID"].nunique()} unique respondents in the train data\nThere are {df_test["RID"].nunique()} unique respondents in the test data')

## MNL

In [ ]:
# Create a Biogeme database from the pandas dataframe
attributes = ['RID','TL1','TT1','Vimg1','TL2','TT2','Vimg2','CHOICE']
biodata_mnl_train = db.Database('cycling_project_roos', df_train[attributes])
biodata_mnl_test = db.Database('cycling_project_roos', df_test[attributes]) 

In [ ]:
# Create biogeme variables
for c in attributes:
    globals()[c] = biodata_mnl_train.variables[c]

In [ ]:
# Give a name to the model    
model_name = 'Linear-additive RUM-MNL'

B_TL = Beta('B_TL', 0, None, None, 0)
B_TT = Beta('B_TT', 0, None, None, 0)
B_IMG = Beta('B_IMG', 1, None, None, 0)

# Utility functions
V1 = B_TL * TL1 + B_TT * TT1 + B_IMG * Vimg1
V2 = B_TL * TL2 + B_TT * TT2 + B_IMG * Vimg2

# Associate utility functions with the numbering of alternatives
V  = {1: V1, 2: V2}

# Associate the availability conditions with the alternatives
AV = {1: 1, 2: 1}

# Estimate the model
results = estimate_mnl(V,AV,CHOICE,biodata_mnl_train,model_name)
sim_results = simulate_mnl(V,AV,CHOICE,biodata_mnl_test,results.get_beta_values(),model_name)

print_results(results)  
print(f'\nTEST SET\nFinal log likelihood:\t{sim_results["LL"]:0.2f}\nRho square:\t\t{sim_results["rho_square"]:0.3f}')

## Mixed Logit

In [ ]:
# Tell Biogeme which variable is the identifier of the individuals
biodata_mnl_train.panel('RID')

# Calculate the number of observations per individual
obs_per_ind = biodata_mnl_train.data['RID'].value_counts().unique()[0]
print(f'Number of observations per individual: {obs_per_ind}')

# Use biogeme's "generateFlatPanelDataFrame to create a wide database in which each row corresponds to one individual
df_wide = biodata_mnl_train.generate_flat_panel_dataframe(identical_columns=None)

# Rename the columns, such that they run from columnname_{0} to columnname_{n} 
renumbered_columns = {col: f'{col.split("_")[1]}_{int(col.split("_")[0])-1}' if len(col.split("_")) == 2 else col for col in df_wide.columns}

# Rename the columns using the dictionary
df_wide.rename(columns=renumbered_columns, inplace=True)

# Create Biogeme database object
biodata_wide_train = db.Database('biodata_mnl_train_wide', df_wide)

# Show the first rows of the wide database
print(f'The wide dataset has a shape of {biodata_wide_train.data.shape}')
biodata_wide_train.data.head()

In [ ]:
# Tell Biogeme which variable is the identifier of the individuals
biodata_mnl_test.panel('RID')

# Calculate the number of observations per individual
obs_per_ind = biodata_mnl_test.data['RID'].value_counts().unique()[0]
print(f'Number of observations per individual: {obs_per_ind}')

# Use biogeme's "generateFlatPanelDataFrame to create a wide database in which each row corresponds to one individual
df_wide = biodata_mnl_test.generate_flat_panel_dataframe(identical_columns=None)

# Rename the columns, such that they run from columnname_{0} to columnname_{n} 
renumbered_columns = {col: f'{col.split("_")[1]}_{int(col.split("_")[0])-1}' if len(col.split("_")) == 2 else col for col in df_wide.columns}

# Rename the columns using the dictionary
df_wide.rename(columns=renumbered_columns, inplace=True)

# Create Biogeme database object
biodata_wide_test = db.Database('biodata_mnl_test_wide', df_wide)

# Show the first rows of the wide database
print(f'The wide dataset has a shape of {biodata_wide_test.data.shape}')
biodata_wide_test.data.head()

In [ ]:
# Create the Biogeme variables for the wide database
TL1   = {q: Variable(f'TL1_{q}')   for q in range(obs_per_ind)}
TT1   = {q: Variable(f'TT1_{q}')   for q in range(obs_per_ind)}
Vimg1 = {q: Variable(f'Vimg1_{q}') for q in range(obs_per_ind)}
TL2   = {q: Variable(f'TL2_{q}')   for q in range(obs_per_ind)}
TT2   = {q: Variable(f'TT2_{q}')   for q in range(obs_per_ind)}
Vimg2 = {q: Variable(f'Vimg2_{q}') for q in range(obs_per_ind)}

### Mixed Logit with Normal Distribution

In [ ]:
# Give the model a name
model_name = 'Panel MXL with normally distributed parameters'

# Parameters definition enabling the construction of random parameters
B_TT     = Beta('B_TT',      -0.2, None, None, 0)
B_TL     = Beta('B_TL',      -0.3, None, None, 0)
B_IMG    = Beta('B_IMG',      1, None, None, 0)
sigma_TT = Beta('sigma_TT',  0.1, None, None, 0)
sigma_TL = Beta('sigma_TL',  0.1, None, None, 0)
sigma_IMG = Beta('sigma_IMG', 0, None, None, 0)

# Construction of random parameters
B_TT_rnd  = B_TT + sigma_TT * bioDraws('B_TT_rnd', 'NORMAL_HALTON2')
B_TL_rnd  = B_TL + sigma_TL * bioDraws('B_TL_rnd', 'NORMAL_HALTON2')
B_IMG_rnd = B_IMG + sigma_IMG * bioDraws('B_IMG_rnd', 'NORMAL_HALTON2')

# Definition of the utility functions
# Note that we use list comprehension to create a list of utility functions for all observations of an individual
V1 = [B_TL_rnd * TL1[q] + B_TT_rnd * TT1[q] + B_IMG_rnd * Vimg1[q] for q in range(obs_per_ind)]
V2 = [B_TL_rnd * TL2[q] + B_TT_rnd * TT2[q] + B_IMG_rnd * Vimg2[q] for q in range(obs_per_ind)]

# Create a dictionary to list the utility functions with the numbering of alternatives
# Note that we use list comprehension to create a list of dictionaries
V = [{1: V1[q], 2: V2[q]} for q in range(obs_per_ind)]
           
# Create a dictionary to describe the availability conditions of each alternative
AV = {1:1, 2:1}

# Specify the number of draws for the Monte Carlo integration
num_draws = 500

# Estimate the model
results = estimate_mxl(V,AV,"CHOICE",obs_per_ind,num_draws, biodata_wide_train, model_name) 
sim_results = simulate_mxl(V,AV,"CHOICE",obs_per_ind,num_draws, biodata_wide_test, results.get_beta_values(), model_name)

print_results(results)
print(f'\nTEST SET\nFinal log likelihood:\t{sim_results["LL"]:0.2f}\nRho square:\t\t{sim_results["rho_square"]:0.3f}')

In [ ]:
# Specify the distribution types for each part of the utility function
distr_types = {
    'TT':  {'dist': 'normal'},
    'TL':  {'dist': 'normal'},
    'IMG': {'dist': 'normal'}
}
# Plot the distributions of the random parameters
xmin = -1
xmax = 3
plot_distributions(results, distr_types, xmin, xmax)

### Mixed Logit with Log-Normal Distributions

In [ ]:
# Give the model a name
model_name = 'Panel MXL with log-normally distributed parameters'

# Parameters definition enabling the construction of random parameters
B_TT     = Beta('B_TT',      -1, None, None, 0)
B_TL     = Beta('B_TL',      -1, None, None, 0)
B_IMG    = Beta('B_IMG',     -3, None, None, 0)
sigma_TT = Beta('sigma_TT',  0.1, None, None, 0)
sigma_TL = Beta('sigma_TL',  0.1, None, None, 0)
sigma_IMG = Beta('sigma_IMG', 0.1, None, None, 0)

# Construction of random parameters
B_TT_rnd  = -exp(B_TT + sigma_TT * bioDraws('B_TT_rnd', 'NORMAL_HALTON2')) # Note the negative sign for TT and TL
B_TL_rnd  = -exp(B_TL + sigma_TL * bioDraws('B_TL_rnd', 'NORMAL_HALTON2'))
B_IMG_rnd = exp(B_IMG + sigma_IMG * bioDraws('B_IMG_rnd', 'NORMAL_HALTON2')) # Note the positive sign for Vimg

# Definition of the utility functions
V1 = [B_TL_rnd * TL1[q] + B_TT_rnd * TT1[q] + B_IMG_rnd * Vimg1[q] for q in range(obs_per_ind)]
V2 = [B_TL_rnd * TL2[q] + B_TT_rnd * TT2[q] + B_IMG_rnd * Vimg2[q] for q in range(obs_per_ind)]

# Create a dictionary to list the utility functions with the numbering of alternatives
V = [{1: V1[q], 2: V2[q]} for q in range(obs_per_ind)]
           
# Create a dictionary to describe the availability conditions of each alternative
AV = {1:1, 2:1}

# Specify the number of draws for the Monte Carlo integration
num_draws = 500  

# Estimate the model
results = estimate_mxl(V,AV,"CHOICE",obs_per_ind,num_draws, biodata_wide_train, model_name) 
sim_results = simulate_mxl(V,AV,"CHOICE",obs_per_ind,num_draws, biodata_wide_test, results.get_beta_values(), model_name)

print_results(results)
print(f'\nTEST SET\nFinal log likelihood:\t{sim_results["LL"]:0.2f}\nRho square:\t\t{sim_results["rho_square"]:0.3f}')

In [ ]:
# Specify the distribution types for each part of the utility function
distr_types = {
    'TT':  {'dist': 'lognormal', 'sign': -1},
    'TL':  {'dist': 'lognormal', 'sign': -1},
    'IMG': {'dist': 'lognormal', 'sign': 1}
}
# Plot the distributions of the random parameters
xmin = -1
xmax = 3
plot_distributions(results, distr_types, xmin, xmax)

### Mixed Logit with log-normal and normal distribution

In [ ]:
# Give the model a name
model_name = 'Panel MXL with normal and log-normally distributed parameters'

# Parameters definition enabling the construction of random parameters
B_TT     = Beta('B_TT',      -0.2, None, None, 0)
B_TL     = Beta('B_TL',      -1, None, None, 0)
B_IMG    = Beta('B_IMG',      1, None, None, 0)
sigma_TT = Beta('sigma_TT',  0.1, None, None, 0)
sigma_TL = Beta('sigma_TL',  0.1, None, None, 0)
sigma_IMG = Beta('sigma_IMG', 0.1, None, None, 0)

# Construction of random parameters
B_TT_rnd  = B_TT + sigma_TT * bioDraws('B_TT_rnd', 'NORMAL_HALTON2') # Normal distribution for TT
B_TL_rnd  = -exp(B_TL + sigma_TL * bioDraws('B_TL_rnd', 'NORMAL_HALTON2')) # Log-normal distribution for TL
B_IMG_rnd = B_IMG + sigma_IMG * bioDraws('B_IMG_rnd', 'NORMAL_HALTON2') 

# Definition of the utility functions
V1 = [B_TL_rnd * TL1[q] + B_TT_rnd * TT1[q] + B_IMG_rnd * Vimg1[q] for q in range(obs_per_ind)]
V2 = [B_TL_rnd * TL2[q] + B_TT_rnd * TT2[q] + B_IMG_rnd * Vimg2[q] for q in range(obs_per_ind)]

# Create a dictionary to list the utility functions with the numbering of alternatives
V = [{1: V1[q], 2: V2[q]} for q in range(obs_per_ind)]
           
# Create a dictionary to describe the availability conditions of each alternative
AV = {1:1, 2:1}

# Specify the number of draws for the Monte Carlo integration
num_draws = 500  

# Estimate the model
results = estimate_mxl(V,AV,"CHOICE",obs_per_ind,num_draws, biodata_wide_train, model_name) 
sim_results = simulate_mxl(V,AV,"CHOICE",obs_per_ind,num_draws, biodata_wide_test, results.get_beta_values(), model_name)

print_results(results)
print(f'\nTEST SET\nFinal log likelihood:\t{sim_results["LL"]:0.2f}\nRho square:\t\t{sim_results["rho_square"]:0.3f}')

In [ ]:
# Specify the distribution types for each part of the utility function
distr_types = {
    'TT':  {'dist': 'normal'},
    'TL':  {'dist': 'lognormal', 'sign': -1},
    'IMG': {'dist': 'normal', 'sign': 1}
}
# Plot the distributions of the random parameters
xmin = -1
xmax = 3
plot_distributions(results, distr_types, xmin, xmax)

In [ ]:
# Give the model a name
model_name = 'Panel MXL with normal and log-normally distributed parameters'

# Parameters definition enabling the construction of random parameters
B_TT     = Beta('B_TT',      -2, None, None, 0)
B_TL     = Beta('B_TL',      -0.2, None, None, 0)
B_IMG    = Beta('B_IMG',      1, None, None, 0)
sigma_TT = Beta('sigma_TT',  0.1, None, None, 0)
sigma_TL = Beta('sigma_TL',  0.1, None, None, 0)
sigma_IMG = Beta('sigma_IMG', 0.1, None, None, 0)

# Construction of random parameters
B_TT_rnd  = -exp(B_TT + sigma_TT * bioDraws('B_TT_rnd', 'NORMAL_HALTON2')) # Log-Normal distribution for TT
B_TL_rnd  = B_TL + sigma_TL * bioDraws('B_TL_rnd', 'NORMAL_HALTON2') # Normal distribution for TL
B_IMG_rnd = B_IMG + sigma_IMG * bioDraws('B_IMG_rnd', 'NORMAL_HALTON2') 

# Definition of the utility functions
V1 = [B_TL_rnd * TL1[q] + B_TT_rnd * TT1[q] + B_IMG_rnd * Vimg1[q] for q in range(obs_per_ind)]
V2 = [B_TL_rnd * TL2[q] + B_TT_rnd * TT2[q] + B_IMG_rnd * Vimg2[q] for q in range(obs_per_ind)]

# Create a dictionary to list the utility functions with the numbering of alternatives
V = [{1: V1[q], 2: V2[q]} for q in range(obs_per_ind)]
           
# Create a dictionary to describe the availability conditions of each alternative
AV = {1:1, 2:1}

# Specify the number of draws for the Monte Carlo integration
num_draws = 500  

# Estimate the model
results = estimate_mxl(V,AV,"CHOICE",obs_per_ind,num_draws, biodata_wide_train, model_name) 
sim_results = simulate_mxl(V,AV,"CHOICE",obs_per_ind,num_draws, biodata_wide_test, results.get_beta_values(), model_name)

print_results(results)
print(f'\nTEST SET\nFinal log likelihood:\t{sim_results["LL"]:0.2f}\nRho square:\t\t{sim_results["rho_square"]:0.3f}')

In [ ]:
# Specify the distribution types for each part of the utility function
distr_types = {
    'TT':  {'dist': 'lognormal', 'sign': -1},
    'TL':  {'dist': 'normal'},
    'IMG': {'dist': 'normal', 'sign': 1}
}
# Plot the distributions of the random parameters
xmin = -1
xmax = 3
plot_distributions(results, distr_types, xmin, xmax)

In [ ]:
# Give the model a name
model_name = 'Panel MXL with normal and log-normally distributed parameters'

# Parameters definition enabling the construction of random parameters
B_TT     = Beta('B_TT',      -1.2, None, None, 0)
B_TL     = Beta('B_TL',      -1.2, None, None, 0)
B_IMG    = Beta('B_IMG',      1, None, None, 0)
sigma_TT = Beta('sigma_TT',  0.1, None, None, 0)
sigma_TL = Beta('sigma_TL',  0.1, None, None, 0)
sigma_IMG = Beta('sigma_IMG', 0.1, None, None, 0)

# Construction of random parameters
B_TT_rnd  = -exp(B_TT + sigma_TT * bioDraws('B_TT_rnd', 'NORMAL_HALTON2')) # Log-Normal distribution for TT
B_TL_rnd  = -exp(B_TL + sigma_TL * bioDraws('B_TL_rnd', 'NORMAL_HALTON2')) # Log-normal distribution for TL
B_IMG_rnd = B_IMG + sigma_IMG * bioDraws('B_IMG_rnd', 'NORMAL_HALTON2') 

# Definition of the utility functions
V1 = [B_TL_rnd * TL1[q] + B_TT_rnd * TT1[q] + B_IMG_rnd * Vimg1[q] for q in range(obs_per_ind)]
V2 = [B_TL_rnd * TL2[q] + B_TT_rnd * TT2[q] + B_IMG_rnd * Vimg2[q] for q in range(obs_per_ind)]

# Create a dictionary to list the utility functions with the numbering of alternatives
V = [{1: V1[q], 2: V2[q]} for q in range(obs_per_ind)]
           
# Create a dictionary to describe the availability conditions of each alternative
AV = {1:1, 2:1}

# Specify the number of draws for the Monte Carlo integration
num_draws = 500  

# Estimate the model
results = estimate_mxl(V,AV,"CHOICE",obs_per_ind,num_draws, biodata_wide_train, model_name) 
sim_results = simulate_mxl(V,AV,"CHOICE",obs_per_ind,num_draws, biodata_wide_test, results.get_beta_values(), model_name)

print_results(results)
print(f'\nTEST SET\nFinal log likelihood:\t{sim_results["LL"]:0.2f}\nRho square:\t\t{sim_results["rho_square"]:0.3f}')

In [ ]:
# Specify the distribution types for each part of the utility function
distr_types = {
    'TT':  {'dist': 'lognormal', 'sign': -1},
    'TL':  {'dist': 'lognormal', 'sign': -1},
    'IMG': {'dist': 'normal', 'sign': 1}
}
# Plot the distributions of the random parameters
xmin = -1
xmax = 3
plot_distributions(results, distr_types, xmin, xmax)